# RMP Liquidity Program & Macro-Valuation Tracking

This notebook implements a framework to track where markets are and where they're going based on the macro-valuation shifts framework. The framework uses two key dimensions:

1. **Yield Curve** (Vertical axis): Steepening vs Flattening
2. **Exchange Rate** (Horizontal axis): Falling vs Rising US Dollar

These are driven by:
- **Capital Flows**: Strong inflows (rightward) vs outflows (leftward)
- **Monetary Policy Stance**: Loose (upward) vs Tight (downward)

## Four Quadrants:
- **Top Right ("Happiness Zone")**: Rising dollar + Steeper yield curve → Strong capital inflows
- **Bottom Right ("Hawkish Policy")**: Rising dollar + Flatter yield curve → Tight monetary conditions
- **Bottom Left ("Crisis Zone")**: Falling dollar + Flatter yield curve → Capital flight
- **Top Left ("Dovish/Co-operative")**: Falling dollar + Steeper yield curve → Loose monetary policy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import yfinance as yf
from fredapi import Fred
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## 1. Key Markets & Indicators to Track

### Primary Indicators:

#### A. Exchange Rate (X-axis)
- **DXY**: US Dollar Index (trade-weighted)
- Alternative: Monitor individual pairs (EUR/USD, USD/JPY, USD/CNY)

#### B. Yield Curve (Y-axis)
- **10Y-2Y Spread**: Primary measure of yield curve shape
- **10Y-3M Spread**: Alternative measure (recession indicator)
- **30Y-5Y Spread**: Long-end curve measure

### Supporting Indicators:

#### C. Capital Flow Metrics
- **TIC Data**: Treasury International Capital flows (monthly, lagged)
- **Foreign Holdings**: Foreign ownership of US Treasuries
- **Net Private Inflows**: Excluding official sector

#### D. Liquidity Conditions (Critical for RMP Program Context)
- **Bank Excess Reserves** (FRED: EXCSRESNW)
- **Fed Balance Sheet** (FRED: WALCL)
- **Reverse Repo (RRP)** (FRED: RRPONTSYD)
- **SOFR**: Secured Overnight Financing Rate
- **Bond Market Fails**: Settlement failures (stress indicator)

#### E. Monetary Policy Stance
- **Fed Funds Rate** (FRED: FEDFUNDS)
- **Fed Balance Sheet Change**: QT vs QE
- **Treasury General Account (TGA)** (FRED: WTREGEN)

#### F. Market Stress Indicators
- **VIX**: Equity volatility
- **MOVE Index**: Bond market volatility
- **IG Spread**: Investment grade credit spread (FRED: BAMLC0A4CBBB)
- **HY Spread**: High yield spread (FRED: BAMLH0A0HYM2)

In [ ]:
# Configuration
# You'll need a FRED API key (free): https://fred.stlouisfed.org/docs/api/api_key.html
FRED_API_KEY = 'YOUR_FRED_API_KEY_HERE'  # Replace with your key

# Date range
start_date = '2020-01-01'
end_date = datetime.today().strftime('%Y-%m-%d')

# Initialize FRED
# fred = Fred(api_key=FRED_API_KEY)

## 2. Data Fetching Functions

In [ ]:
def fetch_market_data(start_date, end_date):
    """
    Fetch key market indicators for macro-valuation tracking.
    
    Returns:
        DataFrame with all key indicators
    """
    
    data = {}
    
    # 1. US Dollar Index (DXY)
    print("Fetching DXY...")
    dxy = yf.download('DX-Y.NYB', start=start_date, end=end_date, progress=False)['Close']
    data['DXY'] = dxy
    
    # 2. Treasury Yields (for yield curve)
    print("Fetching Treasury yields...")
    # 10-year yield
    tnx = yf.download('^TNX', start=start_date, end=end_date, progress=False)['Close']
    data['UST_10Y'] = tnx
    
    # 2-year yield  
    tyx = yf.download('^IRX', start=start_date, end=end_date, progress=False)['Close']
    # Note: ^IRX is 13-week, we'll use TLT/SHY ETF ratio as proxy for now
    # For production, use FRED data: DGS10, DGS2
    
    # 3. VIX
    print("Fetching VIX...")
    vix = yf.download('^VIX', start=start_date, end=end_date, progress=False)['Close']
    data['VIX'] = vix
    
    # 4. Create combined DataFrame
    df = pd.DataFrame(data)
    
    return df


def fetch_fred_data(fred, start_date, end_date):
    """
    Fetch FRED economic data for liquidity and policy tracking.
    
    Parameters:
        fred: Fred API object
        start_date: Start date for data
        end_date: End date for data
        
    Returns:
        DataFrame with FRED indicators
    """
    
    indicators = {
        'EXCSRESNW': 'Bank_Excess_Reserves',
        'WALCL': 'Fed_Balance_Sheet',
        'RRPONTSYD': 'Reverse_Repo',
        'FEDFUNDS': 'Fed_Funds_Rate',
        'WTREGEN': 'Treasury_General_Account',
        'DGS10': 'UST_10Y_Yield',
        'DGS2': 'UST_2Y_Yield',
        'BAMLC0A4CBBB': 'IG_Spread',
        'BAMLH0A0HYM2': 'HY_Spread'
    }
    
    data = {}
    
    for series_id, label in indicators.items():
        try:
            print(f"Fetching {label}...")
            series = fred.get_series(series_id, start_date, end_date)
            data[label] = series
        except Exception as e:
            print(f"Error fetching {label}: {e}")
            data[label] = pd.Series()
    
    df = pd.DataFrame(data)
    
    # Calculate derived metrics
    df['Yield_Curve_10Y2Y'] = df['UST_10Y_Yield'] - df['UST_2Y_Yield']
    df['Net_Liquidity'] = (df['Fed_Balance_Sheet'] 
                           - df['Reverse_Repo'] 
                           - df['Treasury_General_Account'])
    
    return df

## 3. Quadrant Positioning Calculations

In [ ]:
def calculate_quadrant_position(df, lookback_period=252):
    """
    Calculate position in the macro-valuation quadrant system.
    
    Parameters:
        df: DataFrame with market indicators
        lookback_period: Period for z-score normalization (default: 252 days)
        
    Returns:
        DataFrame with quadrant coordinates (x, y)
    """
    
    result = df.copy()
    
    # X-axis: Exchange Rate (DXY)
    # Normalize to z-score over lookback period
    result['DXY_zscore'] = (
        (result['DXY'] - result['DXY'].rolling(lookback_period).mean()) / 
        result['DXY'].rolling(lookback_period).std()
    )
    
    # Y-axis: Yield Curve (10Y-2Y spread)
    result['YieldCurve_zscore'] = (
        (result['Yield_Curve_10Y2Y'] - result['Yield_Curve_10Y2Y'].rolling(lookback_period).mean()) / 
        result['Yield_Curve_10Y2Y'].rolling(lookback_period).std()
    )
    
    # Assign quadrants
    def assign_quadrant(row):
        x, y = row['DXY_zscore'], row['YieldCurve_zscore']
        
        if pd.isna(x) or pd.isna(y):
            return 'Unknown'
        
        if x >= 0 and y >= 0:
            return 'Happiness Zone'
        elif x >= 0 and y < 0:
            return 'Hawkish Policy'
        elif x < 0 and y < 0:
            return 'Crisis Zone'
        else:
            return 'Dovish/Co-operative'
    
    result['Quadrant'] = result.apply(assign_quadrant, axis=1)
    
    return result


def calculate_momentum_indicators(df, short_window=20, long_window=60):
    """
    Calculate momentum to determine directional movement in quadrant space.
    
    Parameters:
        df: DataFrame with quadrant positions
        short_window: Short-term moving average window
        long_window: Long-term moving average window
        
    Returns:
        DataFrame with momentum indicators
    """
    
    result = df.copy()
    
    # DXY momentum (x-axis movement)
    result['DXY_momentum'] = (
        result['DXY'].rolling(short_window).mean() - 
        result['DXY'].rolling(long_window).mean()
    )
    
    # Yield curve momentum (y-axis movement)
    result['YieldCurve_momentum'] = (
        result['Yield_Curve_10Y2Y'].rolling(short_window).mean() - 
        result['Yield_Curve_10Y2Y'].rolling(long_window).mean()
    )
    
    # Net liquidity momentum (predictive for future moves)
    result['NetLiquidity_momentum'] = result['Net_Liquidity'].pct_change(30)
    
    return result

## 4. Visualization Functions

In [ ]:
def plot_quadrant_spider(df, recent_months=36, annotation_dates=None):
    """
    Create spider diagram showing path through macro-valuation quadrants.
    
    Parameters:
        df: DataFrame with quadrant positions
        recent_months: Number of recent months to highlight
        annotation_dates: List of dates to annotate (optional)
    """
    
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # Filter data
    cutoff_date = df.index[-1] - pd.DateOffset(months=recent_months)
    df_recent = df[df.index >= cutoff_date].copy()
    
    # Plot historical path (faded)
    df_historical = df[df.index < cutoff_date].copy()
    ax.plot(df_historical['DXY_zscore'], df_historical['YieldCurve_zscore'], 
            'o-', alpha=0.2, color='gray', markersize=2, label='Historical')
    
    # Plot recent path (colorful gradient)
    points = ax.scatter(df_recent['DXY_zscore'], df_recent['YieldCurve_zscore'],
                       c=range(len(df_recent)), cmap='plasma', s=50, 
                       alpha=0.7, edgecolors='black', linewidth=0.5)
    
    # Connect recent points
    ax.plot(df_recent['DXY_zscore'], df_recent['YieldCurve_zscore'], 
            '-', alpha=0.4, color='blue', linewidth=1.5)
    
    # Mark current position
    current = df.iloc[-1]
    ax.scatter(current['DXY_zscore'], current['YieldCurve_zscore'], 
              s=300, marker='*', color='red', edgecolors='black', 
              linewidth=2, zorder=5, label='Current')
    
    # Add quadrant labels
    quadrant_labels = [
        (1.5, 1.5, "Happiness Zone\n(Strong Inflows)", 'green'),
        (1.5, -1.5, "Hawkish Policy\n(Tight Monetary)", 'orange'),
        (-1.5, -1.5, "Crisis Zone\n(Capital Flight)", 'red'),
        (-1.5, 1.5, "Dovish/Co-op\n(Loose Policy)", 'blue')
    ]
    
    for x, y, label, color in quadrant_labels:
        ax.text(x, y, label, fontsize=11, ha='center', va='center',
               bbox=dict(boxstyle='round', facecolor=color, alpha=0.2))
    
    # Add axis lines
    ax.axhline(y=0, color='black', linestyle='-', linewidth=1, alpha=0.3)
    ax.axvline(x=0, color='black', linestyle='-', linewidth=1, alpha=0.3)
    
    # Labels
    ax.set_xlabel('Exchange Rate (DXY) →\nFalling ← | → Rising', fontsize=12, fontweight='bold')
    ax.set_ylabel('Yield Curve (10Y-2Y) →\nFlattening ← | → Steepening', fontsize=12, fontweight='bold')
    ax.set_title('Macro-Valuation Quadrant Spider Diagram\nCapital Flows & Monetary Policy Positioning', 
                fontsize=14, fontweight='bold', pad=20)
    
    # Add colorbar for time
    cbar = plt.colorbar(points, ax=ax)
    cbar.set_label('Time (Recent to Latest)', rotation=270, labelpad=20)
    
    # Grid
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(loc='upper left')
    
    # Set reasonable axis limits
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    
    plt.tight_layout()
    plt.show()


def plot_liquidity_dashboard(df):
    """
    Create comprehensive liquidity dashboard for RMP program context.
    """
    
    fig, axes = plt.subplots(3, 2, figsize=(16, 12))
    
    # 1. Bank Excess Reserves
    ax = axes[0, 0]
    ax.plot(df.index, df['Bank_Excess_Reserves'], linewidth=2, color='blue')
    ax.fill_between(df.index, df['Bank_Excess_Reserves'], alpha=0.3, color='blue')
    ax.set_title('Bank Excess Reserves (Critical Liquidity Measure)', fontweight='bold')
    ax.set_ylabel('Billions USD')
    ax.grid(True, alpha=0.3)
    
    # 2. Fed Balance Sheet vs RRP
    ax = axes[0, 1]
    ax.plot(df.index, df['Fed_Balance_Sheet']/1000, label='Fed Balance Sheet', linewidth=2)
    ax.plot(df.index, df['Reverse_Repo']/1000, label='Reverse Repo', linewidth=2)
    ax.set_title('Fed Balance Sheet vs Reverse Repo', fontweight='bold')
    ax.set_ylabel('Trillions USD')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 3. Net Liquidity (Fed BS - RRP - TGA)
    ax = axes[1, 0]
    ax.plot(df.index, df['Net_Liquidity']/1000, linewidth=2, color='green')
    ax.fill_between(df.index, df['Net_Liquidity']/1000, alpha=0.3, color='green')
    ax.set_title('Net Liquidity (Fed BS - RRP - TGA)', fontweight='bold')
    ax.set_ylabel('Trillions USD')
    ax.grid(True, alpha=0.3)
    
    # 4. Yield Curve
    ax = axes[1, 1]
    ax.plot(df.index, df['Yield_Curve_10Y2Y'], linewidth=2, color='purple')
    ax.axhline(y=0, color='red', linestyle='--', linewidth=1, alpha=0.5, label='Inversion')
    ax.fill_between(df.index, df['Yield_Curve_10Y2Y'], 0, 
                     where=df['Yield_Curve_10Y2Y']>0, alpha=0.3, color='green', label='Steep')
    ax.fill_between(df.index, df['Yield_Curve_10Y2Y'], 0, 
                     where=df['Yield_Curve_10Y2Y']<=0, alpha=0.3, color='red', label='Inverted')
    ax.set_title('Yield Curve (10Y-2Y Spread)', fontweight='bold')
    ax.set_ylabel('Basis Points')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 5. Credit Spreads
    ax = axes[2, 0]
    ax.plot(df.index, df['IG_Spread'], label='IG Spread', linewidth=2)
    ax.plot(df.index, df['HY_Spread'], label='HY Spread', linewidth=2)
    ax.set_title('Credit Spreads (Stress Indicators)', fontweight='bold')
    ax.set_ylabel('Basis Points')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 6. DXY
    ax = axes[2, 1]
    ax.plot(df.index, df['DXY'], linewidth=2, color='darkgreen')
    ax.fill_between(df.index, df['DXY'], alpha=0.3, color='darkgreen')
    ax.set_title('US Dollar Index (DXY)', fontweight='bold')
    ax.set_ylabel('Index Level')
    ax.grid(True, alpha=0.3)
    
    plt.suptitle('Liquidity & Macro Dashboard (RMP Context)', 
                fontsize=16, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.show()


def plot_quadrant_history(df):
    """
    Plot time series showing which quadrant the market has been in.
    """
    
    fig, ax = plt.subplots(figsize=(14, 6))
    
    # Map quadrants to numeric values for plotting
    quadrant_map = {
        'Happiness Zone': 3,
        'Hawkish Policy': 2,
        'Crisis Zone': 1,
        'Dovish/Co-operative': 4,
        'Unknown': 0
    }
    
    colors = {
        3: 'green',
        2: 'orange',
        1: 'red',
        4: 'blue',
        0: 'gray'
    }
    
    df['Quadrant_Numeric'] = df['Quadrant'].map(quadrant_map)
    
    # Plot as colored areas
    for quadrant, color in colors.items():
        mask = df['Quadrant_Numeric'] == quadrant
        if mask.any():
            ax.fill_between(df.index, 0, 1, where=mask, alpha=0.5, 
                           color=color, transform=ax.get_xaxis_transform())
    
    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='green', alpha=0.5, label='Happiness Zone'),
        Patch(facecolor='orange', alpha=0.5, label='Hawkish Policy'),
        Patch(facecolor='red', alpha=0.5, label='Crisis Zone'),
        Patch(facecolor='blue', alpha=0.5, label='Dovish/Co-operative')
    ]
    
    ax.legend(handles=legend_elements, loc='upper left', ncol=4)
    ax.set_title('Macro-Valuation Quadrant History', fontsize=14, fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_yticks([])
    ax.grid(True, axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 5. Example Usage & Market Tracking

**Note**: To run this notebook, you need:
1. A free FRED API key from https://fred.stlouisfed.org/docs/api/api_key.html
2. Internet connection to fetch data

### Step-by-step:

In [ ]:
# Step 1: Fetch market data (yfinance - no API key needed)
print("Fetching market data...")
market_data = fetch_market_data(start_date, end_date)
print(f"Fetched {len(market_data)} days of market data")
market_data.tail()

In [ ]:
# Step 2: Fetch FRED data (requires API key)
# Uncomment and run after setting FRED_API_KEY above

# print("Fetching FRED economic data...")
# fred = Fred(api_key=FRED_API_KEY)
# fred_data = fetch_fred_data(fred, start_date, end_date)
# print(f"Fetched {len(fred_data)} days of FRED data")

# # Merge datasets
# combined_data = pd.merge(market_data, fred_data, 
#                          left_index=True, right_index=True, how='outer')
# combined_data = combined_data.sort_index()
# combined_data.tail()

In [ ]:
# Step 3: Calculate quadrant positions
# After merging data, calculate positions

# df_with_quadrants = calculate_quadrant_position(combined_data)
# df_with_quadrants = calculate_momentum_indicators(df_with_quadrants)

# print("\nCurrent Market Position:")
# current = df_with_quadrants.iloc[-1]
# print(f"Quadrant: {current['Quadrant']}")
# print(f"DXY Z-Score: {current['DXY_zscore']:.2f}")
# print(f"Yield Curve Z-Score: {current['YieldCurve_zscore']:.2f}")
# print(f"DXY Level: {current['DXY']:.2f}")
# print(f"10Y-2Y Spread: {current['Yield_Curve_10Y2Y']:.0f} bps")

In [ ]:
# Step 4: Visualize quadrant spider diagram
# plot_quadrant_spider(df_with_quadrants, recent_months=36)

In [ ]:
# Step 5: Liquidity dashboard
# plot_liquidity_dashboard(df_with_quadrants)

In [ ]:
# Step 6: Quadrant history
# plot_quadrant_history(df_with_quadrants)

## 6. Interpreting the Current Position

### Current Market Analysis Framework:

**What to monitor daily/weekly:**

1. **Position Changes**: Has the market moved quadrants?
2. **Momentum Direction**: Which direction is momentum pointing?
3. **Liquidity Stress**: Are reserves declining? Are credit spreads widening?
4. **Policy Response**: How is the Fed responding (RMP program, repo operations)?

**Implications by Quadrant:**

- **Happiness Zone** (Top Right): 
  - Strong USD ✓
  - Steeper curve ✓
  - Capital flowing in ✓
  - Risk assets supported
  - Examples: 1984, 2000, 2015, 2020

- **Hawkish Policy** (Bottom Right):
  - Strong USD ✓
  - Flatter curve ✗
  - Tight monetary conditions ✗
  - Liquidity draining
  - **This is where RMP interventions become critical**

- **Crisis Zone** (Bottom Left):
  - Weak USD ✗
  - Flatter curve ✗
  - Capital flight ✗✗
  - Emergency measures needed
  - Examples: 1980, 1990, 2010

- **Dovish/Co-operative** (Top Left):
  - Weak USD ✗
  - Steeper curve ✓
  - Easy money ✓
  - Reflation trades

### Key Watchpoints for 2026:

Per the article's forecast:
- DXY expected to **firm by ~5%** in 2026
- Movement toward **bottom-right quadrant** (Hawkish Policy)
- Implies: Rising USD + Flatter curves
- Driver: Tightening liquidity despite RMP 'sticking plaster'

**Critical Thresholds to Monitor:**
- Bank Excess Reserves < $3T (danger zone)
- 10Y-2Y spread < -50bps (deep inversion) or rapid steepening
- VIX > 30 (sustained stress)
- IG Spreads > 200bps (credit concerns)
- DXY > 110 (strong dollar stress on EMs)

## 7. Real-Time Tracking Checklist

### Daily Monitoring:
- [ ] DXY level and trend
- [ ] 10Y-2Y spread
- [ ] VIX level
- [ ] SOFR (funding stress)
- [ ] Repo rates

### Weekly Monitoring:
- [ ] Fed balance sheet (Wednesday release)
- [ ] Reverse Repo usage
- [ ] Bank excess reserves
- [ ] Credit spreads (IG/HY)
- [ ] Treasury auction results

### Monthly Monitoring:
- [ ] TIC data (capital flows)
- [ ] Fed policy minutes
- [ ] Foreign holdings of UST
- [ ] Trade balance

### Key Data Sources:
- **FRED**: https://fred.stlouisfed.org/
- **TIC Data**: https://home.treasury.gov/data/treasury-international-capital-tic-system
- **Fed H.4.1**: https://www.federalreserve.gov/releases/h41/
- **NY Fed**: https://www.newyorkfed.org/markets/desk-operations
- **Bloomberg**: Terminal access for real-time data
- **Yahoo Finance**: Free market data (limited fundamentals)

## 8. Forward-Looking Projections

Based on the article's framework, we can project potential paths:

### Scenario Analysis:

In [ ]:
def project_dxy_scenario(current_dxy, expected_change_pct, horizon_days=365):
    """
    Project DXY path based on expected % change.
    
    Per article: DXY expected to firm by ~5% in 2026
    """
    target_dxy = current_dxy * (1 + expected_change_pct/100)
    
    # Create projection path with some volatility
    dates = pd.date_range(start=datetime.today(), periods=horizon_days, freq='D')
    
    # Linear path with random walk
    trend = np.linspace(current_dxy, target_dxy, horizon_days)
    noise = np.random.normal(0, current_dxy * 0.003, horizon_days).cumsum()
    projection = trend + noise
    
    return pd.Series(projection, index=dates)


# Example projection
# current_dxy = market_data['DXY'].iloc[-1]
# dxy_projection = project_dxy_scenario(current_dxy, expected_change_pct=5)

# plt.figure(figsize=(12, 6))
# plt.plot(market_data.index, market_data['DXY'], label='Historical', linewidth=2)
# plt.plot(dxy_projection.index, dxy_projection, label='Projected (+5%)', 
#          linestyle='--', linewidth=2, color='red')
# plt.axhline(y=current_dxy, color='gray', linestyle=':', alpha=0.5)
# plt.fill_between(dxy_projection.index, 
#                  dxy_projection * 0.98, 
#                  dxy_projection * 1.02,
#                  alpha=0.2, color='red', label='±2% Band')
# plt.title('DXY Projection for 2026 (Article Baseline: +5%)', fontsize=14, fontweight='bold')
# plt.xlabel('Date')
# plt.ylabel('DXY Level')
# plt.legend()
# plt.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.show()

## 9. Summary: Markets to Track

### Essential Tracking (Minimum viable):
1. **DXY** (US Dollar Index) - X-axis positioning
2. **10Y-2Y Spread** - Y-axis positioning
3. **Bank Excess Reserves** - Liquidity health
4. **VIX** - Stress gauge

### Recommended Tracking (Comprehensive):
5. **Fed Balance Sheet** - Policy stance
6. **Reverse Repo (RRP)** - Liquidity absorption
7. **Credit Spreads** (IG/HY) - Risk pricing
8. **SOFR** - Funding costs
9. **TIC Flows** - Capital movements
10. **Fed Funds Rate** - Policy rate

### Advanced Tracking (Professional):
11. **Treasury General Account (TGA)** - Fiscal drain
12. **Foreign UST Holdings** - Demand source
13. **MOVE Index** - Bond volatility
14. **Cross-currency basis swaps** - Dollar funding stress
15. **Central bank swap lines usage** - Offshore stress

---

## Next Steps:
1. Get a FRED API key (free)
2. Run the data fetching code
3. Generate current quadrant position
4. Set up weekly monitoring routine
5. Track movement relative to 2026 forecast (rightward + downward = Hawkish Policy quadrant)